# FinAgent — ReAct loop walk-through

This notebook shows the **exact internals** of the agent. We don't use any framework: we manually drive the loop

```
user query → model → (tool_calls?) → execute tools → feed results back → model → final answer
```

Requires a GPU (the fine-tuned 7B model loads via unsloth in 4-bit). On a single A100/L40S, one query takes ~3–6 s end-to-end.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

from agent_from_scratch import load_model, generate, parse_tool_calls, execute_tools
from configs.prompt_templates import SYSTEM_PROMPT

model, tokenizer = load_model('danab17/finagent-7b-merged')

## Step 1 — Build the initial messages

Just two messages: the system prompt (FinAgent's identity + guardrails) and the user query.

In [ ]:
query = "Compare Apple and Microsoft on valuation, and tell me which looks more attractive."
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': query},
]
print(json.dumps(messages, indent=2)[:800])

## Step 2 — First model turn

The model should emit a `<think>` block, then `[TOOL_CALLS]` with one or more tool calls in parallel.

In [ ]:
response_1 = generate(model, tokenizer, messages)
print(response_1)

In [ ]:
tool_calls = parse_tool_calls(response_1)
tool_calls

## Step 3 — Run the tools

`execute_tools` looks up each name in `TOOL_REGISTRY` and runs the function with the parsed arguments.

In [ ]:
tool_results = execute_tools(tool_calls)
for r in tool_results:
    print(r['name'], '→', json.dumps(r['result'])[:200], '...')

## Step 4 — Append tool results and ask the model again

We feed the assistant message (with tool_calls) and each tool response back into `messages`. The model now has data to ground its answer.

In [ ]:
messages.append({'role': 'assistant', 'content': response_1})
for r in tool_results:
    messages.append({'role': 'tool', 'name': r['name'], 'content': json.dumps(r['result'])})

response_2 = generate(model, tokenizer, messages)
print(response_2)

## Step 5 — Is there a tool call in turn 2?

If not → we have our final answer. If yes → loop. Most queries finish in 1 model-tool-model round trip.

In [ ]:
more_calls = parse_tool_calls(response_2)
print('more tool calls?' , bool(more_calls))

That's it — that's the entire ReAct loop. The `react_loop()` function in `agent_from_scratch.py` just wraps the four steps above in a `for _ in range(max_iters)` loop, and the LangGraph version does the same with a state graph instead.